# 96장 · 출처 표기 함수와 응답 카드

출처 표기는 읽는 사람이 원문으로 돌아갈 만큼 구체적이어야 합니다. 문서 ID 외에 원문의 위치를 조합해 답변 카드에 표시합니다.

Q01의 검색 결과로 화면에 표시할 답변과 원문 위치를 조립합니다. 장비명이 없는 질문은 규정을 임의로 고르는 대신 추가 확인 경로로 보냅니다.

실행을 확인한 커널은 Python 3.12.15입니다.

Python 3 커널을 선택하고 위에서부터 실행하세요. 팀별 변형을 시도한 뒤 해당 셀 아래에서 예시를 펼쳐 비교할 수 있습니다.


## 검색 결과로 원문 카드를 조립하기

95장에서 실제로 찾은 문단으로 카드의 본문과 위치를 채웁니다. 검색어를 바꿔 후보가 사라지면 빈 화면이 어떻게 표시되는지도 확인할 수 있습니다.

이 프로젝트의 노트북은 한 폴더에서 열어 주세요. 첫 셀은 `team-project/final/chapter95.json`을 먼저 찾습니다. 팀 파일이 없으면 [이 장의 시작 예시](chapter96-start.json)를 노트북 옆에 내려받으세요. 출력 첫 줄에서 어느 파일을 읽었는지 확인할 수 있습니다.

In [ ]:
from pathlib import Path
import json
import re

here = Path.cwd()
project_root = here.parent if re.fullmatch(r"chapter[0-9]+", here.name) else here
team_file = project_root / "team-project/final/chapter95.json"
sample_file = here / "chapter96-start.json"
input_file = team_file if team_file.is_file() else sample_file
if not input_file.is_file():
    raise FileNotFoundError("이 장의 시작 예시를 내려받아 노트북 옆에 두세요.")
incoming = json.loads(input_file.read_text(encoding="utf-8"))
print("팀 기록" if input_file == team_file else "교재 예시", input_file.name)
print(incoming['hits'])

교재 예시 chapter96-start.json
[{'id': 'RULE-A', 'text': '점검 전 전원 표시등을 확인합니다.', 'source': '가상 장비실 규정 A', 'page': 1, 'section': '2.1', 'revised_on': '2026-11-01', 'version': 'rules-v1', 'equipment': '장비 A', 'search_text': '점검 전 전원 표시등을 확인합니다.', 'sha256': '8336aeeaa41f3a89559a2364950ef88f7d6128639fb3dad46974a7be38ac2f42'}]


두 번째 줄은 읽은 파일의 `hits` 값입니다.

## citation_line()으로 원문 위치 표시

첫 수용 사례　질문 “장비 A의 점검 전 확인 사항은?” / 답 “점검 전 전원 표시등을 확인합니다.” / 근거 RULE-A, 가상 장비실 규정 A, 1쪽 2.1절, 2026-11-01, rules-v1. 응답 데이터를 그대로 보여 주면 어느 숫자가 페이지인지 알기 어렵습니다. 화면 문장에 단위를 붙이되 문서의 실제 값은 바꾸지 않습니다.


### 셀 01 · citation_line()의 위치 표기

로컬 규칙 Q01의 저장 응답을 읽어 출처를 화면용 문자열로 만듭니다.


In [1]:
reply = {'status': 'answered',
 'answer': '점검 전 전원 표시등을 확인합니다.',
 'sources': [{'id': 'RULE-A',
              'text': '점검 전 전원 표시등을 확인합니다.',
              'source': '가상 장비실 규정 A',
              'page': 1,
              'section': '2.1',
              'revised_on': '2026-11-01',
              'version': 'rules-v1'}],
 'mode': 'local_rule_fixture',
 'model_calls': 0}
def citation_line(source):
    return (f"{source['id']} | {source['source']} | p.{source['page']} | "
            f"{source['section']}절 | 개정일 {source['revised_on']} | {source['version']}")

def show_reply(response):
    print(response["answer"])
    for source in response["sources"]:
        print(citation_line(source))

show_reply(reply)

점검 전 전원 표시등을 확인합니다.
RULE-A | 가상 장비실 규정 A | p.1 | 2.1절 | 개정일 2026-11-01 | rules-v1


답 바로 아래에 RULE-A의 위치가 붙습니다. 출처 ID만 나왔다면 검색 결과의 누락인지 화면 함수에서 일부 필드만 읽었는지 두 값을 대조하세요.

인용의 각 필드는 원문을 다시 찾는 데 쓰입니다. RULE-A로 자료를 식별하고 1쪽 2.1절로 이동한 뒤 개정일을 대조합니다. 다른 조가 이 정보만으로 시연에 사용한 문장을 찾을 수 있는지가 출처 표시의 확인 기준입니다.


### show_reply()의 추가 질문

아래 셀을 실행한 뒤 바로 앞의 결과와 비교하세요.


In [2]:
reply_missing = {"answer": "장비 A인가요, 장비 B인가요?", "sources": []}
show_reply(reply_missing)

장비 A인가요, 장비 B인가요?


출처가 비어 있어서 안내 문장 한 줄만 나옵니다. 앞 질문의 RULE-A를 다시 표시하지 않습니다.


#### 근거 카드를 그리기 전에

<mark>장비를 특정할 수 없는 질문에는 먼저 대상을 물어봅니다.</mark> 장비 A의 근거를 한 번 표시한 다음 장비명이 빠진 질문을 보내세요. 새 확인 질문 아래에 방금 전 규정의 쪽·절이 남는지 보면 상태 전환을 검증할 수 있습니다.

선수 개념 · 88장 인수 기준과 작업 분해

#### 출처가 0건이면 show_reply()에서 어느 줄까지 실행될까?

```mermaid
flowchart TD
nreply["응답 받기"]
  nprint["답 표시"]
  nloop["출처 순회"]
  ncitation["인용 줄"]
  nend["표시 종료"]
  nreply --> nprint
  nprint --> nloop
  nloop --> ncitation
  ncitation --> nloop
  nloop --> nend
```

추가 질문도 답변 영역에 표시됩니다. 빈 sources를 순회하면 citation_line()은 한 번도 호출되지 않습니다.

## show_reply()의 추가 질문 분기

Q02의 완료 예시　상태 need_clarification / 문구 “장비 A 또는 장비 B 중 하나를 지정하세요.” / 출처 없음. 출처가 항상 한 건이라고 가정하고 목록의 첫 항목을 읽으면 이 화면에서는 오류가 납니다. 답변 영역은 먼저 상태를 읽고, 출처가 있을 때만 원문 위치를 그립니다.


### 팀 참고 예시 · 출처 카드 변형


In [3]:
reply_b = {'status': 'answered',
 'answer': '점검 전 고정 장치를 확인합니다.',
 'sources': [{'id': 'RULE-B',
              'text': '점검 전 고정 장치를 확인합니다.',
              'source': '가상 장비실 규정 B',
              'page': 2,
              'section': '2.2',
              'revised_on': '2026-11-01',
              'version': 'rules-v1'}],
 'mode': 'local_rule_fixture',
 'model_calls': 0}
print(reply_b["answer"])
print(citation_line(reply_b["sources"][0]))

점검 전 고정 장치를 확인합니다.
RULE-B | 가상 장비실 규정 B | p.2 | 2.2절 | 개정일 2026-11-01 | rules-v1


고정 장치, RULE-B, 2쪽 2.2절로 바뀝니다. A의 페이지나 절이 그대로면 화면에 적힌 고정 문구부터 찾으세요.


## 근거 카드의 표시와 원문 데이터를 서로 검산하기

RULE-A 카드는 눈으로 보기에 완성됐습니다. RULE-B를 받았을 때도 쪽·절이 A에 고정되어 있지 않은지 같은 표시 함수를 통해 확인합니다.

### 셀 05 · 받은 레코드로 두 번째 카드 조립하기

앞에서 작성한 citation_line에 RULE-B의 실제 출처를 건넵니다. 표시한 문자열에 필요한 값이 들어가는지 전달 데이터와 대조합니다.

In [4]:
current_source = reply_b["sources"][0]
visible_citation = citation_line(current_source)
assert current_source["id"] == "RULE-B"
assert f'p.{current_source["page"]}' in visible_citation
assert f'{current_source["section"]}절' in visible_citation
print(reply_b["answer"])
print(visible_citation)

점검 전 고정 장치를 확인합니다.
RULE-B | 가상 장비실 규정 B | p.2 | 2.2절 | 개정일 2026-11-01 | rules-v1


2쪽 2.2절이 보이고 문장도 고정 장치를 가리킵니다. 출처만 B로 바꾸고 답이 A의 전원 표시등으로 남으면 페이지 검사만으로는 발견하지 못합니다.

### 동료가 카드에서 확인할 두 지점

<table><thead><tr><th scope="col">입력</th><th scope="col">답변에서 짚을 단어</th><th scope="col">원문에서 펼칠 위치</th></tr></thead><tbody><tr><td>장비 A의 점검 전 확인 사항은?</td><td>전원 표시등</td><td>가상 규정 A 1쪽 2.1절</td></tr><tr><td>장비 B의 점검 전 확인 사항은?</td><td>고정 장치</td><td>가상 규정 B 2쪽 2.2절</td></tr></tbody></table>

한 사람은 질문을 고르고 다른 사람은 답변 카드만 보고 원문을 찾아갑니다. 팀의 자료에서도 이름이 비슷하되 위치와 핵심 문장이 다른 두 문서를 골라 보세요.

<mark>정상 카드의 정답에는 답의 핵심 단어와 실제로 열리는 위치가 모두 들어갑니다.</mark> B 카드에 RULE-B가 보여도 답이 전원 표시등이면 서로 다른 문서가 섞인 것입니다.

97장에는 A 정상 답 뒤에 자료 없는 질문과 B 정상 답을 잇는 세 턴을 전달합니다.

## 팀 사례에 적용하기 · 검색 결과로 원문 카드를 조립하기

위 예제에 첫 셀에서 읽어 둔 팀 기록을 넣어 봅니다.

In [ ]:
cards = [{"id": doc["id"], "answer": doc["text"], "citation": citation_line(doc)}
         for doc in incoming["hits"]]
new = {"quality_reference": incoming["quality_reference"], "cards": cards, "prepared": incoming["prepared"]}
print([card["citation"] for card in cards])

['RULE-A | 가상 장비실 규정 A | p.1 | 2.1절 | 개정일 2026-11-01 | rules-v1']


citation_line에 넣은 레코드와 답에 사용한 레코드가 같아서, 문장과 출처가 같은 판본을 가리킵니다. 화면에 다른 장비의 자료가 나타나면 두 값이 어디서 갈라졌는지 보세요.

In [ ]:
output_file = project_root / "team-project/final/chapter96.json"
output_file.parent.mkdir(parents=True, exist_ok=True)
with output_file.open("a+", encoding="utf-8") as handle:
    handle.seek(0)
    previous_text = handle.read()
    if previous_text and json.loads(previous_text) != new:
        raise FileExistsError("같은 장의 다른 결과가 있습니다. 기존 JSON을 다른 이름으로 보관한 뒤 다시 실행하세요.")
    if not previous_text:
        json.dump(new, handle, ensure_ascii=False, indent=2)
        handle.write("\n")
print(output_file.relative_to(project_root).as_posix())

team-project/final/chapter96.json


97장 첫 셀이 이 파일을 읽습니다. 입력을 바꿔 다시 실행할 때는 기존 결과 파일의 이름을 바꿔 보관하세요.

## 돌아보기

Q01의 답과 원문 위치, Q02에서 다시 묻는 문장을 97장에 넘기세요. 화면 시안의 출처를 보고 다른 사람이 실제 절을 찾을 수 있는지는 별도 관찰 항목입니다.
